<a href="https://colab.research.google.com/github/Jayaprakasam1412/customer_segmentation-CLV-Dashboard/blob/main/customer_segmentation_clv_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**load Data**

In [ ]:
import pandas as pd
df = pd.read_csv("ecommerce_rfm_dataset.csv")
df

,InvoiceNo,CustomerID,ProductID,InvoiceDate,Quantity,UnitPrice,Amount,Region
0,INV00000,CUST0103,PROD021,2024-06-03,3,335.27,1005.81,West
1,INV00001,CUST0436,PROD019,2023-04-08,4,448.45,1793.80,West
2,INV00002,CUST0349,PROD053,2023-08-24,4,291.09,1164.36,South
3,INV00003,CUST0271,PROD019,2024-01-09,1,266.06,266.06,West
4,INV00004,CUST0107,PROD057,2024-08-03,2,84.39,168.78,North
...,...,...,...,...,...,...,...,...
4995,INV04995,CUST0401,PROD078,2024-07-01,1,43.00,43.00,North
4996,INV04996,CUST0233,PROD097,2024-02-21,4,158.57,634.28,South
4997,INV04997,CUST0158,PROD092,2023-06-08,2,255.81,511.62,West
4998,INV04998,CUST0437,PROD052,2023-05-29,4,294.41,1177.64,North


In [ ]:
df.head()

,InvoiceNo,CustomerID,ProductID,InvoiceDate,Quantity,UnitPrice,Amount,Region
0,INV00000,CUST0103,PROD021,2024-06-03,3,335.27,1005.81,West
1,INV00001,CUST0436,PROD019,2023-04-08,4,448.45,1793.80,West
2,INV00002,CUST0349,PROD053,2023-08-24,4,291.09,1164.36,South
3,INV00003,CUST0271,PROD019,2024-01-09,1,266.06,266.06,West
4,INV00004,CUST0107,PROD057,2024-08-03,2,84.39,168.78,North


**Understand Data**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    5000 non-null   object 
 1   CustomerID   5000 non-null   object 
 2   ProductID    5000 non-null   object 
 3   InvoiceDate  5000 non-null   object 
 4   Quantity     5000 non-null   int64  
 5   UnitPrice    5000 non-null   float64
 6   Amount       5000 non-null   float64
 7   Region       5000 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 312.6+ KB


In [ ]:
df.describe()

,Quantity,UnitPrice,Amount
count,5000.000000,5000.000000,5000.000000
mean,2.511000,253.261134,635.447676
std,1.117197,141.321339,476.168435
min,1.000000,5.060000,5.320000
25%,2.000000,131.455000,258.900000
50%,3.000000,253.575000,494.795000
75%,4.000000,371.627500,922.050000
max,4.000000,499.950000,1999.800000


**Data Cleaning**

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
df.isnull().sum()

,0
InvoiceNo,0
CustomerID,0
ProductID,0
InvoiceDate,0
Quantity,0
UnitPrice,0
Amount,0
Region,0


**Create "Today Date"**

In [ ]:
import datetime as dt

today = dt.datetime(2025,1,1)

**RFM Calculation (CORE)**

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (today - x.max()).days,
    'InvoiceNo': 'count',
    'Amount': 'sum'
})

rfm.columns = ['Recency','Frequency','Monetary']
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
CUST0001,30,14,10527.24
CUST0002,199,10,6519.48
CUST0003,337,9,3445.64
CUST0004,59,10,5245.03
CUST0005,89,15,7856.83


**RFM SCORING (1–5)**

In [ ]:
rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F'] = pd.qcut(rfm['Frequency'], 5, labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm.head()

,Recency,Frequency,Monetary,R,F,M
CustomerID,,,,,,
CUST0001,30,14,10527.24,4,5,5
CUST0002,199,10,6519.48,1,3,3
CUST0003,337,9,3445.64,1,2,1
CUST0004,59,10,5245.03,3,3,2
CUST0005,89,15,7856.83,2,5,4


**CREATE CUSTOMER SEGMENTS**

In [ ]:
def segment(row):
    if row['R'] == 5 and row['F'] >= 4 and row['M'] >= 4:
        return "Champions"
    elif row['F'] >= 4:
        return "Loyal Customers"
    elif row['R'] <= 2:
        return "At Risk"
    elif row['R'] == 1:
        return "Lost"
    else:
        return "Others"

rfm['Segment'] = rfm.apply(segment, axis=1)

rfm.head()

,Recency,Frequency,Monetary,R,F,M,Segment
CustomerID,,,,,,,
CUST0001,30,14,10527.24,4,5,5,Loyal Customers
CUST0002,199,10,6519.48,1,3,3,At Risk
CUST0003,337,9,3445.64,1,2,1,At Risk
CUST0004,59,10,5245.03,3,3,2,Others
CUST0005,89,15,7856.83,2,5,4,Loyal Customers


**CHECK SEGMENT DISTRIBUTION**

In [ ]:
rfm['Segment'].value_counts()

,count
Segment,
Others,180
At Risk,157
Loyal Customers,126
Champions,36


**EXPORT FOR POWER BI**

In [ ]:
df.to_csv("rfm_output.csv", index=False)
from google.colab import files
files.download("rfm_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
rfm.head()

,Recency,Frequency,Monetary,R,F,M,Segment
CustomerID,,,,,,,
CUST0001,30,14,10527.24,4,5,5,Loyal Customers
CUST0002,199,10,6519.48,1,3,3,At Risk
CUST0003,337,9,3445.64,1,2,1,At Risk
CUST0004,59,10,5245.03,3,3,2,Others
CUST0005,89,15,7856.83,2,5,4,Loyal Customers


In [ ]:
rfm.to_csv("rfm_output.csv", index=True)

In [ ]:
from google.colab import files
files.download("rfm_output.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>